# Hybrid Token S-Adapter (ConvNeXt-Base + ViT-Base) for Face Anti-Spoofing
## Implementasi Lengkap Dual-Branch Token Cross-Attention Versi BASE Sesuai Gambar Arsitektur

Notebook ini mengimplementasikan varian **BASE** dari arsitektur **Face Anti-Spoofing (FAS)** modern sesuai `Arsitektur TA.png`:
1. **Branch 1 (ConvNeXt-Base)**: Ekstraksi fitur mikro tekstur lokal $\rightarrow$ Output Stage 4 $(B, C_4=1024, 7, 7) \rightarrow$ Spatial Flatten $49$ tokens $(B, 49, 1024) \rightarrow$ **ConvNeXt Token S-Adapter (TSR via Depthwise Conv)** dengan proyeksi linear $1024 \rightarrow d=768 \rightarrow$ Local Tokens $T_c$ $(B, 49, 768)$.
2. **Branch 2 (ViT-Base)**: Ekstraksi konteks global $\rightarrow$ Patch Embedding $16\times 16 \rightarrow 196$ patch tokens $(B, 196, 768) \rightarrow 12$ Transformer Blocks dengan **S-Adapter (PEFT)** $\rightarrow$ Buang [CLS] Token $\rightarrow$ Global Tokens $T_v$ $(B, 196, 768)$.
3. **Fusion & Classification Module**:
   - **Bidirectional Cross-Attention (12 Heads)**: Local-to-Global ($Q=T_c, K=T_v, V=T_v$) dan Global-to-Local ($Q=T_v, K=T_c, V=T_c$) pada dimensi $d=768$.
   - **Concatenate & Refine**: $T_r = [T'_c \; ; \; T'_v]$ berukuran $(B, 245, 768) \rightarrow$ Transformer Refinement Block (12 Heads, MLP).
   - **Global Pooling & Classifier**: Average Pooling $\rightarrow (B, 768) \rightarrow$ Linear($768 \rightarrow 256$) $\rightarrow$ ReLU $\rightarrow$ Dropout(0.3) $\rightarrow$ Linear($256 \rightarrow 2$) $\rightarrow$ Output (Real / Spoof).


In [ ]:
# ============================================================
# 1. SETUP ENVIRONMENT, LIBRARIES & DEVICE
# ============================================================
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import math, random, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.io import read_image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

import timm
from timm import create_model

# Reproducibility Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device yang digunakan: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ============================================================
# 2. MEMBANGUN MANIFEST DATASET CELEBA-SPOOF
# ============================================================
def build_manifest(data_root="./CelebA_Spoof"):
    data_root = Path(data_root)
    records = []
    for split in ['train', 'test']:
        split_dir = data_root / split
        if not split_dir.exists():
            split_dir = data_root / 'Data' / split
        if not split_dir.exists():
            continue
        for subj in split_dir.iterdir():
            if not subj.is_dir(): continue
            for t in subj.iterdir():
                if not t.is_dir(): continue
                # Label 1 = Live (Asli), Label 0 = Spoof (Palsu)
                label = 1 if t.name.lower() in ['1', 'live'] else 0
                for img in list(t.glob('*.jpg')) + list(t.glob('*.png')):
                    records.append({
                        'path': str(img),
                        'label': label,
                        'split': split,
                        'subject': subj.name
                    })
    return pd.DataFrame(records)

print("Mencari gambar CelebA-Spoof...")
df_full = build_manifest("./CelebA_Spoof")
print(f"Total foto ditemukan: {len(df_full):,} ({sum(df_full.label==1):,} Live, {sum(df_full.label==0):,} Spoof)")


In [ ]:
# ============================================================
# 3. PEMBAGIAN DATA SUBJECT-INDEPENDENT (TRAIN 64%, VAL 16%, TEST 20%)
# ============================================================
subjs = list(df_full.subject.unique())
if len(subjs) >= 3:
    train_subjs, test_subjs = train_test_split(subjs, test_size=0.2, random_state=SEED)
    train_subjs, val_subjs  = train_test_split(train_subjs, test_size=0.2, random_state=SEED)
    df_train = df_full[df_full.subject.isin(train_subjs)].reset_index(drop=True)
    df_val   = df_full[df_full.subject.isin(val_subjs)].reset_index(drop=True)
    df_test  = df_full[df_full.subject.isin(test_subjs)].reset_index(drop=True)
else:
    df_train, df_test = train_test_split(df_full, test_size=0.2, random_state=SEED)
    df_train, df_val  = train_test_split(df_train, test_size=0.2, random_state=SEED)

print(f"Split: Train={len(df_train):,} | Val={len(df_val):,} | Test={len(df_test):,}")


In [ ]:
# ============================================================
# 4. TRANSFORMS & DATASET DENGAN RETINAFACE BBOX SCALING
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(3, sigma=(0.1, 1.0)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CelebASpoofDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, use_bbox: bool = True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_bbox = use_bbox

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = Path(row.path)
        try:
            img = read_image(str(img_path))
            if img.shape[0] == 4: img = img[:3]
            if img.shape[0] == 1: img = img.repeat(3, 1, 1)

            if self.use_bbox:
                bb_path = Path(str(img_path).replace('.jpg', '_BB.txt').replace('.png', '_BB.txt'))
                if bb_path.exists():
                    with open(bb_path, 'r') as f:
                        parts = f.read().strip().split()
                        if len(parts) >= 4:
                            bbox = [float(p) for p in parts]
                            real_h, real_w = img.shape[1], img.shape[2]
                            x1 = max(0, int(bbox[0] * (real_w / 224.0)))
                            y1 = max(0, int(bbox[1] * (real_h / 224.0)))
                            w1 = int(bbox[2] * (real_w / 224.0))
                            h1 = int(bbox[3] * (real_h / 224.0))
                            x2 = min(real_w, x1 + w1)
                            y2 = min(real_h, y1 + h1)
                            if (x2 - x1) > 10 and (y2 - y1) > 10:
                                img = img[:, y1:y2, x1:x2]
        except Exception:
            img = torch.zeros(3, IMG_SIZE, IMG_SIZE, dtype=torch.uint8)

        if self.transform:
            img = self.transform(img.numpy().transpose(1, 2, 0))

        return img, int(row.label)

def make_weighted_sampler(df: pd.DataFrame):
    counts = df.label.value_counts().to_dict()
    total = len(df)
    weights = [total / counts[lbl] for lbl in df.label]
    return WeightedRandomSampler(weights, len(weights), replacement=True)

# Batch size 16 optimal untuk model Base di GPU VRAM 18GB
BATCH_SIZE = 16
train_ds = CelebASpoofDataset(df_train, train_transform, use_bbox=True)
val_ds   = CelebASpoofDataset(df_val, val_transform, use_bbox=True)
test_ds  = CelebASpoofDataset(df_test, val_transform, use_bbox=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=make_weighted_sampler(df_train), num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"DataLoaders Siap: train={len(train_loader)} batches | val={len(val_loader)} | test={len(test_loader)}")


In [ ]:
# ============================================================
# 5. DEFINISI ARSITEKTUR HYBRID TOKEN S-ADAPTER (VERSI BASE)
#    ConvNeXt-Base (C4=1024) + ViT-Base (d=768)
# ============================================================

# ── 1. S-Adapter untuk Vision Transformer (PEFT) ──
class SAdapter(nn.Module):
    def __init__(self, d_model: int = 768, bottleneck_dim: int = 64):
        super().__init__()
        self.down_proj = nn.Linear(d_model, bottleneck_dim)
        self.act       = nn.GELU()
        self.up_proj   = nn.Linear(bottleneck_dim, d_model)
        self.scale     = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.scale * self.up_proj(self.act(self.down_proj(x)))

# ── 2. S-Adapter untuk ConvNeXt-Base Tokens (TSR via Depthwise Conv) ──
class ConvNeXtTokenSAdapter(nn.Module):
    def __init__(self, in_channels: int = 1024, embed_dim: int = 768, bottleneck_dim: int = 64, spatial_size: int = 7):
        super().__init__()
        self.spatial_size = spatial_size
        # 2D Local Statistical Adapter (TSR)
        self.adapter_2d = nn.Sequential(
            nn.Conv2d(in_channels, bottleneck_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(bottleneck_dim),
            nn.GELU(),
            nn.Conv2d(bottleneck_dim, bottleneck_dim, kernel_size=3, padding=1, groups=bottleneck_dim, bias=False),
            nn.BatchNorm2d(bottleneck_dim),
            nn.GELU(),
            nn.Conv2d(bottleneck_dim, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.scale = nn.Parameter(torch.ones(1) * 0.1)
        # Proyeksi linear dari C4=1024 ke d=768
        self.proj = nn.Linear(in_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        B, N, C = tokens.shape
        H = W = self.spatial_size
        feat_2d = tokens.permute(0, 2, 1).reshape(B, C, H, W)
        adapted_2d = self.adapter_2d(feat_2d)
        adapted_tokens = adapted_2d.flatten(2).permute(0, 2, 1)
        x = tokens + self.scale * adapted_tokens
        return self.norm(self.proj(x))

# ── 3. ConvNeXt-Base Branch (Local Feature Extraction) ──
class ConvNeXtTokenBranch(nn.Module):
    def __init__(self, model_name: str = "convnext_base", embed_dim: int = 768, adapter_bottleneck: int = 64, pretrained: bool = True, freeze_backbone: bool = True):
        super().__init__()
        base = create_model(model_name, pretrained=pretrained, num_classes=0)
        self.stem = base.stem
        self.stages = base.stages
        self.norm = base.norm_pre
        self.out_channels = base.num_features  # 1024 untuk convnext_base

        if freeze_backbone:
            for p in base.parameters(): p.requires_grad = False
            for p in self.stages[-1].parameters(): p.requires_grad = True

        self.convnext_adapter = ConvNeXtTokenSAdapter(
            in_channels=self.out_channels,
            embed_dim=embed_dim,
            bottleneck_dim=adapter_bottleneck,
            spatial_size=7
        )

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        x = self.stages(x)
        feat_map = self.norm(x)  # (B, 1024, 7, 7)
        tokens_49 = feat_map.flatten(2).permute(0, 2, 1)  # (B, 49, 1024)
        Tc = self.convnext_adapter(tokens_49)             # (B, 49, 768)
        return Tc, feat_map

# ── 4. ViT-Base Branch (Global Feature Extraction) ──
class ViTWithSAdapter(nn.Module):
    def __init__(self, vit_model_name: str = "vit_base_patch16_224", adapter_bottleneck: int = 64, pretrained: bool = True, freeze_backbone: bool = True):
        super().__init__()
        self.vit = create_model(vit_model_name, pretrained=pretrained, num_classes=0)
        self.embed_dim = self.vit.embed_dim  # 768 untuk ViT-Base
        self.adapters = nn.ModuleList([SAdapter(self.embed_dim, adapter_bottleneck) for _ in range(len(self.vit.blocks))])

        if freeze_backbone:
            for param in self.vit.parameters(): param.requires_grad = False
            for param in self.adapters.parameters(): param.requires_grad = True

    def forward(self, x: torch.Tensor):
        x = self.vit.patch_embed(x)             # (B, 196, 768)
        cls_token = self.vit.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)    # (B, 197, 768)
        x = self.vit.pos_drop(x + self.vit.pos_embed)
        for block, adapter in zip(self.vit.blocks, self.adapters):
            x = adapter(block(x))
        x = self.vit.norm(x)
        Tv = x[:, 1:, :]  # Buang CLS token -> Global Tokens Tv (B, 196, 768)
        return Tv

# ── 5. Fusion Module: Bidirectional Cross-Attention (12 Heads) ──
class BidirectionalTokenCrossAttention(nn.Module):
    def __init__(self, embed_dim: int = 768, num_heads: int = 12, dropout: float = 0.1):
        super().__init__()
        self.local_to_global_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.global_to_local_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_tc1 = nn.LayerNorm(embed_dim); self.norm_tc2 = nn.LayerNorm(embed_dim)
        self.norm_tv1 = nn.LayerNorm(embed_dim); self.norm_tv2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, Tc: torch.Tensor, Tv: torch.Tensor):
        q_c, k_v = self.norm_tc1(Tc), self.norm_tv1(Tv)
        l2g, _ = self.local_to_global_attn(query=q_c, key=k_v, value=k_v)
        T_prime_c = self.norm_tc2(Tc + self.dropout(l2g))

        q_v, k_c = self.norm_tv1(Tv), self.norm_tc1(Tc)
        g2l, _ = self.global_to_local_attn(query=q_v, key=k_c, value=k_c)
        T_prime_v = self.norm_tv2(Tv + self.dropout(g2l))
        return T_prime_c, T_prime_v

# ── 6. Transformer Refinement Block & Classifier Head ──
class TokenRefinementBlock(nn.Module):
    def __init__(self, embed_dim: int = 768, num_heads: int = 12, mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim), nn.Dropout(dropout)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class TokenAntiSpoofingClassifier(nn.Module):
    def __init__(self, embed_dim: int = 768, num_classes: int = 2, dropout: float = 0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)

# ── 7. Model Hybrid Lengkap Versi BASE ──
class HybridTokenSAdapterConvNeXtBase(nn.Module):
    def __init__(self, adapter_bottleneck: int = 64, pretrained: bool = True, freeze_backbone: bool = True):
        super().__init__()
        self.embed_dim = 768
        self.vit_branch = ViTWithSAdapter("vit_base_patch16_224", adapter_bottleneck, pretrained=pretrained, freeze_backbone=freeze_backbone)
        self.convnext_branch = ConvNeXtTokenBranch("convnext_base", embed_dim=self.embed_dim, adapter_bottleneck=adapter_bottleneck, pretrained=pretrained, freeze_backbone=freeze_backbone)
        self.cross_attention = BidirectionalTokenCrossAttention(embed_dim=self.embed_dim, num_heads=12)
        self.refinement_block = TokenRefinementBlock(embed_dim=self.embed_dim, num_heads=12)
        self.classifier = TokenAntiSpoofingClassifier(embed_dim=self.embed_dim)

    def forward(self, x: torch.Tensor):
        Tc, _ = self.convnext_branch(x)                # (B, 49, 768)
        Tv = self.vit_branch(x)                        # (B, 196, 768)
        T_prime_c, T_prime_v = self.cross_attention(Tc, Tv)
        Tr = torch.cat([T_prime_c, T_prime_v], dim=1)  # (B, 245, 768)
        Tr_refined = self.refinement_block(Tr)          # (B, 245, 768)
        pooled = Tr_refined.mean(dim=1)                 # (B, 768)
        logits = self.classifier(pooled)                # (B, 2)
        probs = F.softmax(logits, dim=-1)
        return {"logits": logits, "probs": probs, "Tc": Tc, "Tv": Tv, "Tr": Tr_refined, "pooled": pooled}


In [ ]:
# ============================================================
# 6. VERIFIKASI DIMENSI TENSOR MODEL BASE DENGAN ARSITEKTUR TA.PNG
# ============================================================
model_base = HybridTokenSAdapterConvNeXtBase(adapter_bottleneck=64, pretrained=False).to(DEVICE)

total_params = sum(p.numel() for p in model_base.parameters())
trainable_params = sum(p.numel() for p in model_base.parameters() if p.requires_grad)

print(f"--- SPESIFIKASI MODEL HYBRID BASE ---")
print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print(f"Shared Dimension (d)  : {model_base.embed_dim}")

x_dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    out = model_base(x_dummy)

print("\n--- VALIDASI BENTUK TENSOR TIAP TAHAPAN SESUAI ARSITEKTUR TA.PNG ---")
print(f"1. Input Image              : {x_dummy.shape}")
print(f"2. Local Tokens Tc (ConvNeXt): {out['Tc'].shape}   -> Sesuai (B, 49, d=768)")
print(f"3. Global Tokens Tv (ViT)   : {out['Tv'].shape}  -> Sesuai (B, 196, d=768)")
print(f"4. Refined Tokens Tr (Fused): {out['Tr'].shape}  -> Sesuai (B, 245, d=768)")
print(f"5. Global Pooled Token      : {out['pooled'].shape}     -> Sesuai (B, d=768)")
print(f"6. Output Logits (Live/Spoof): {out['logits'].shape}       -> Sesuai (B, 2)")
print("\n[SUCCESS] Seluruh dimensi model BASE 100% PERSIS SESUAI Arsitektur TA.png!")


In [ ]:
# ============================================================
# 7. FOCAL LOSS & OPTIMIZER KHUSUS BASE
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        focal = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal.mean()

criterion = FocalLoss(gamma=2.0, alpha=0.25)

adapter_params = [p for n, p in model_base.named_parameters() if ("adapter" in n or "stages.3" in n) and p.requires_grad]
head_params    = [p for n, p in model_base.named_parameters() if ("adapter" not in n and "stages.3" not in n) and p.requires_grad]

optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": 1e-4, "weight_decay": 1e-4},
    {"params": head_params, "lr": 1e-3, "weight_decay": 1e-2},
])

EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
print("Focal Loss, Optimizer & Scheduler untuk BASE Siap!")


In [ ]:
# ============================================================
# 8. METRIK EVALUASI FAS & FUNGSI EVALUATE
# ============================================================
def calculate_metrics(y_true, y_scores, threshold=0.5):
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    y_pred = (y_scores >= threshold).astype(int)

    spoof_mask = (y_true == 0)
    apcer = float(np.mean(y_pred[spoof_mask] == 1)) if spoof_mask.sum() > 0 else 0.0

    live_mask = (y_true == 1)
    bpcer = float(np.mean(y_pred[live_mask] == 0)) if live_mask.sum() > 0 else 0.0

    hter = (apcer + bpcer) / 2.0
    acc = float(np.mean(y_pred == y_true))

    try: auc = roc_auc_score(y_true, y_scores)
    except: auc = 0.5

    return {"accuracy": acc, "auc": auc, "apcer": apcer, "bpcer": bpcer, "hter": hter}

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_targets, all_scores = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            out = model(images)
            loss = criterion(out["logits"], labels)
            total_loss += loss.item() * images.size(0)
            all_targets.extend(labels.cpu().numpy().tolist())
            all_scores.extend(out["probs"][:, 1].cpu().numpy().tolist())

    metrics = calculate_metrics(all_targets, all_scores)
    metrics["loss"] = total_loss / len(all_targets)
    return metrics, all_targets, all_scores


In [ ]:
# ============================================================
# 9. TRAINING & VALIDASI KHUSUS BASE
# ============================================================
Path("./checkpoints_base").mkdir(parents=True, exist_ok=True)
BEST_BASE_CKPT = Path("./checkpoints_base/best_token_hybrid_base_model.pth")

history_base = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_auc": [], "val_hter": []}
best_val_auc = 0.0

print("=" * 75)
print("MEMULAI TRAINING HYBRID CONVNEXT-BASE + VIT-BASE S-ADAPTER")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    model_base.train()
    running_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Base Train]")
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model_base(images)
        loss = criterion(out["logits"], labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = out["logits"].argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{100*correct/total:.1f}%"})

    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total

    val_metrics, _, _ = evaluate(model_base, val_loader)
    elapsed = time.time() - t0

    history_base["train_loss"].append(train_loss)
    history_base["train_acc"].append(train_acc)
    history_base["val_loss"].append(val_metrics["loss"])
    history_base["val_acc"].append(val_metrics["accuracy"])
    history_base["val_auc"].append(val_metrics["auc"])
    history_base["val_hter"].append(val_metrics["hter"])

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_metrics['loss']:.4f} Acc: {val_metrics['accuracy']*100:.2f}% "
          f"AUC: {val_metrics['auc']:.4f} HTER: {val_metrics['hter']*100:.2f}%")

    if val_metrics["auc"] > best_val_auc:
        best_val_auc = val_metrics["auc"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": model_base.state_dict(),
            "val_auc": best_val_auc,
            "val_acc": val_metrics["accuracy"],
            "val_hter": val_metrics["hter"],
            "embed_dim": 768,
            "arch": "HybridTokenSAdapterConvNeXt_BASE"
        }, BEST_BASE_CKPT)
        print(f"  -> Checkpoint BASE terbaik disimpan di: {BEST_BASE_CKPT}")


In [ ]:
# ============================================================
# 10. VISUALISASI GRAFIK HASIL TRAINING & VALIDASI (BASE)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history_base["train_loss"], label="Train Loss", color="blue", linewidth=2)
axes[0].plot(history_base["val_loss"], label="Val Loss", color="red", linewidth=2)
axes[0].set_title("BASE: Training & Validation Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history_base["train_acc"], label="Train Acc", color="blue", linewidth=2)
axes[1].plot(history_base["val_acc"], label="Val Acc", color="green", linewidth=2)
axes[1].set_title("BASE: Training & Validation Accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history_base["val_auc"], label="Val AUC", color="purple", linewidth=2)
axes[2].plot(history_base["val_hter"], label="Val HTER", color="orange", linewidth=2)
axes[2].set_title("BASE: Validation AUC & HTER", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./training_history_token_hybrid_base.png", dpi=150)
plt.show()
print("[Success] Grafik training BASE tersimpan di ./training_history_token_hybrid_base.png")


In [ ]:
# ============================================================
# 11. EVALUASI FINAL PADA TEST SET & PLOT ROC CURVE (BASE)
# ============================================================
if BEST_BASE_CKPT.exists():
    ckpt = torch.load(BEST_BASE_CKPT, map_location=DEVICE)
    model_base.load_state_dict(ckpt["model_state_dict"])
    print(f"Checkpoint BASE terbaik (Epoch {ckpt['epoch']}) dimuat!")

test_metrics, test_targets, test_scores = evaluate(model_base, test_loader)

print("\n" + "="*65)
print("HASIL EVALUASI FINAL TEST SET (HYBRID CONVNEXT-BASE + VIT-BASE)")
print("="*65)
print(f"Accuracy : {test_metrics['accuracy']*100:.2f}%")
print(f"AUC      : {test_metrics['auc']:.4f}")
print(f"APCER    : {test_metrics['apcer']*100:.2f}%  (False Acceptance Spoof)")
print(f"BPCER    : {test_metrics['bpcer']*100:.2f}%  (False Rejection Live)")
print(f"HTER     : {test_metrics['hter']*100:.2f}%  (Half Total Error Rate)")
print("="*65)

fpr, tpr, _ = roc_curve(test_targets, test_scores)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {test_metrics["auc"]:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (APCER)')
plt.ylabel('True Positive Rate (1 - BPCER)')
plt.title('ROC Curve Test Set - Hybrid Model (BASE)', fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()
